# Experiment A: Hyperparameter Tuning with Optuna

이 노트북은 베스트 특성 세트(Advanced FE)를 기반으로 `Optuna`를 활용하여 베이지안 최적화를 수행합니다.
CatBoost 모델의 잠재력을 최대한 끌어올리기 위해 학습률, 트리 깊이 등의 하이퍼파라미터를 자동으로 탐색합니다.

## 1. 환경 설정 및 데이터 로드

In [ ]:
import pandas as pd
import numpy as np
import optuna
from catboost import CatBoostClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
import gc

DATA_PATH = '/kaggle/input/competitions/playground-series-s6e5/'
train = pd.read_csv(DATA_PATH + 'train.csv')
test = pd.read_csv(DATA_PATH + 'test.csv')
submission = pd.read_csv(DATA_PATH + 'sample_submission.csv')

print(f'Train Shape: {train.shape}, Test Shape: {test.shape}')

## 2. Advanced Feature Engineering
EDA를 통해 도출된 핵심 인사이트(타이어 상대 수명, 경기 후반부, 마모 가속도)를 특성으로 추가합니다.

In [ ]:
def advanced_engineering(df):
    compound_mean_life = {'HARD': 25, 'MEDIUM': 18, 'SOFT': 12, 'INTERMEDIATE': 20, 'WET': 15}
    df['Relative_TyreLife'] = df['TyreLife'] / df['Compound'].map(compound_mean_life).fillna(20)
    df['Is_Final_Laps'] = (df['RaceProgress'] > 0.85).astype(int)
    df['Degradation_Momentum'] = df['TyreLife'] * df['Cumulative_Degradation']
    return df

train = advanced_engineering(train)
test = advanced_engineering(test)

cat_features = ['Driver', 'Compound', 'Race', 'Year']
drop_cols = ['id', 'PitNextLap']
features = [c for c in train.columns if c not in drop_cols]

X = train[features]
y = train['PitNextLap']
groups = train['Race']

## 3. Optuna 목적 함수 (Objective Function) 정의
Optuna가 최적화할 대상(AUC)과 탐색할 파라미터 공간을 설정합니다.

In [ ]:
def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 800, 1500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 8),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'random_strength': trial.suggest_float('random_strength', 1e-3, 10.0, log=True),
        'eval_metric': 'AUC',
        'verbose': 0,
        'random_seed': 42,
        'task_type': 'CPU' # Kaggle GPU 사용시 'GPU'로 변경 권장
    }
    
    kf = GroupKFold(n_splits=3) # 속도를 위해 3-Fold로 탐색
    oof_preds = np.zeros(len(X))
    
    for train_idx, val_idx in kf.split(X, y, groups):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model = CatBoostClassifier(**params)
        model.fit(X_train, y_train, cat_features=cat_features, eval_set=(X_val, y_val), early_stopping_rounds=30)
        
        oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
        
    return roc_auc_score(y, oof_preds)

## 4. Optuna 스터디 실행
실제로 파라미터 공간을 탐색하여 최적의 조합을 찾습니다. (`n_trials`를 조절하여 탐색 횟수를 변경할 수 있습니다.)

In [ ]:
print("Starting Optuna tuning...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

print('\n--- Optuna Tuning Results ---')
print('Best trial parameters:', study.best_trial.params)
print('Best AUC Score:', study.best_value)

## 5. 최적 파라미터로 최종 모델 학습 및 제출
찾아낸 베스트 파라미터를 사용하여 전체 5-Fold 검증을 수행하고 제출 파일을 생성합니다.

In [ ]:
best_params = study.best_trial.params
best_params.update({'eval_metric': 'AUC', 'verbose': 100, 'random_seed': 42})

kf = GroupKFold(n_splits=5)
test_preds = np.zeros(len(test))

print("\nTraining final model with best parameters...")
for train_idx, val_idx in kf.split(X, y, groups):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = CatBoostClassifier(**best_params)
    model.fit(X_train, y_train, cat_features=cat_features, eval_set=(X_val, y_val), early_stopping_rounds=50)
    test_preds += model.predict_proba(test[features])[:, 1] / 5

submission['PitNextLap'] = test_preds
submission.to_csv('submission_exp_A.csv', index=False)
print("\nSubmission file 'submission_exp_A.csv' saved successfully.")